# Schematic2SDN — Vision Model Fine-Tuning (Colab)

This notebook trains a 4-bit LoRA adapter over **Qwen2.5-VL** using the audited, leak-resistant Schematic2SDN paired circuit dataset.

### Recommended GPU:
- **T4 GPU**: uses `--profile 3b-8gb` (Qwen2.5-VL-3B, ~6.5 GiB VRAM floor)
- **A100 / L4 GPU**: uses `--profile 7b` (Qwen2.5-VL-7B, ~12 GiB VRAM floor)

## 1. Check GPU Environment

In [ ]:
!nvidia-smi
import torch
print('PyTorch version:', torch.__version__)
print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Name:', torch.cuda.get_device_name(0))
    print('BF16 Supported:', torch.cuda.is_bf16_supported())
    print(f'Free VRAM: {torch.cuda.mem_get_info()[0] / (1024**3):.2f} GiB')

## 2. Mount Google Drive & Upload/Clone Repo
Mount your drive to save checkpoints directly to persistent storage.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Set up workspace
!mkdir -p /content/workspace
%cd /content/workspace

## 3. Extract Codebase & Prepared Dataset
Upload `prepared_amsnet.zip` (15 MB) to Colab or Google Drive, then unzip.

In [ ]:
# Copy or unzip prepared_amsnet.zip
!unzip -q /content/prepared_amsnet.zip -d /content/dataset
# Check split records
!wc -l /content/dataset/training/prepared_amsnet/*.jsonl

## 4. Install Training Dependencies

In [ ]:
!pip install --upgrade pip
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install transformers peft bitsandbytes accelerate pillow pydantic fastapi uvicorn httpx

## 5. Preflight Smoke Test (Memory & Headroom Check)
Runs 1 optimizer step on the longest circuit annotation to verify that GPU memory does not exceed bounds.

In [ ]:
# For T4 GPU:
!python training/finetune.py \
    --profile 3b-8gb \
    --dataset /content/dataset/training/prepared_amsnet \
    --output /content/checkpoints/circuit-lora \
    --smoke-test

# OR for A100 / L4 GPU:
# !python training/finetune.py \
#     --profile 7b \
#     --dataset /content/dataset/training/prepared_amsnet \
#     --output /content/checkpoints/circuit-lora \
#     --smoke-test

## 6. Run Fine-Tuning

In [ ]:
# Train with 3 epochs (change profile to '7b' if running on A100/L4)
!python training/finetune.py \
    --profile 3b-8gb \
    --dataset /content/dataset/training/prepared_amsnet \
    --output /content/checkpoints/circuit-lora \
    --epochs 3

## 7. Evaluate on Held-Out Test Set

In [ ]:
# Launch local inference server in background
import subprocess, time
!S2SDN_ADAPTER=/content/checkpoints/circuit-lora/adapter S2SDN_MODEL_TAG=schematic2sdn-qwen-vl python -m uvicorn serve:app --app-dir training --host 127.0.0.1 --port 11435 &
time.sleep(10)

# Run evaluation
!IMAGE_MODE=vision VISION_MODEL=schematic2sdn-qwen-vl VISION_OLLAMA_URL=http://127.0.0.1:11435 python training/evaluate.py \
    --dataset /content/dataset/training/prepared_amsnet \
    --output /content/vision-evaluation.json \
    --limit 64

## 8. Save Checkpoints to Google Drive

In [ ]:
!cp -r /content/checkpoints/circuit-lora /content/drive/MyDrive/Schematic2SDN_circuit_lora
!cp /content/vision-evaluation.json /content/drive/MyDrive/
print('[+] Saved model adapter and evaluation report to Google Drive!')